# 🧠 Day 2 — Language AI: LLMs, Prompting & RAG
### AI Workshop · Google Colab Notebook

---

**Today you will:**
- Run a Large Language Model (LLM) and understand how it generates text
- Write prompts that control model behaviour precisely
- Build a complete RAG (Retrieval-Augmented Generation) chatbot
- Understand *why* LLMs hallucinate and *how* RAG fixes it

**The pipeline today:**
```
Document (DATA) → Embedding Model (MODEL) → Vectors (OUTPUT)
                                                  ↓
Question (DATA) → Retrieve chunks → LLM (MODEL) → Answer (OUTPUT)
```

> 💡 **This notebook runs 100% on Google Colab — no installs, no API keys, no local setup.**  
> All models download automatically to Colab's cloud VM.  
> Run cells **in order from top to bottom**.  
> If anything breaks: **Runtime → Restart session → Run all**

> ⏱️ **First run:** Section 0 takes 3–5 minutes (model downloads ~2.2 GB). After that, every cell is fast.


---
## Section 0 — Setup
Run all cells in this section before anything else.

In [1]:
# Install all required packages
# Run this first — takes about 60–90 seconds
!pip install chromadb sentence-transformers transformers accelerate -q

print('✅ All packages installed.')


✅ All packages installed.


In [2]:
# ── LOAD THE LLM — runs entirely inside Colab, zero API key needed ────────
# Model: TinyLlama-1.1B-Chat — instruction-tuned, fast, fits Colab free tier
# Downloads ~2.2 GB the first time; cached on subsequent runs

from transformers import pipeline
import torch

# Detect GPU (faster) or fall back to CPU
device = 0 if torch.cuda.is_available() else -1
dtype  = torch.float16 if device == 0 else torch.float32

if device == 0:
    print('🟢 GPU detected — model will run fast')
else:
    print('🟡 No GPU found — running on CPU (slower)')
    print('   Tip: Runtime → Change runtime type → T4 GPU → Save')

print('\nDownloading the model')

llm_pipe = pipeline(
    'text-generation',
    model='ibm-granite/granite-4.1-3b',
    torch_dtype=dtype,
    device=device,
)

print('\n✅ LLM loaded and ready!')


🟢 GPU detected — model will run fast



`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]


✅ LLM loaded and ready!


In [3]:
# ── call_llm() — our unified LLM interface for the whole notebook ────────
# Every exercise calls this function — no config switches, no tokens

def call_llm(prompt, system_prompt='You are a helpful assistant.', temperature=0.7):
    """
    Send a prompt+system_prompt to TinyLlama and return the reply as a string.

    Parameters
    ----------
    prompt        : The user question or task.
    system_prompt : Persona / rules for the model (invisible to end user).
    temperature   : 0.1 = focused/predictable, 1.4 = creative/random.
    """
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': prompt},
    ]
    # TinyLlama uses a specific chat template — apply it before sending
    formatted = llm_pipe.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    result = llm_pipe(
        formatted,
        max_new_tokens=700,
        temperature=max(float(temperature), 0.01),
        do_sample=(temperature > 0.05),
        pad_token_id=llm_pipe.tokenizer.eos_token_id,
        return_full_text=False,
    )
    return result[0]['generated_text'].strip()

print('✅ call_llm() defined. Run the next cell to test.')


✅ call_llm() defined. Run the next cell to test.


In [4]:
# Quick sanity check — confirm the LLM is working before exercises
print('Testing LLM...')

test_response = call_llm(
    prompt='Say exactly this sentence and nothing else: I am ready to teach AI today.',
    temperature=0.1
)
print(f'\n✅ LLM responded:')
print(f'   {test_response}')
print('\nEverything is working. Move on to Section 1!')


Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'pad_token_id', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Testing LLM...

✅ LLM responded:
   I am ready to teach AI today.

Everything is working. Move on to Section 1!


---
## Section 1 — How LLMs Work: The Core Intuition

Before we use an LLM, let's build an intuition for what's happening inside it.

### Key concepts:

| Concept | Plain English |
|---|---|
| **Token** | A word piece (~¾ of a word on average). "AI" = 1 token. "transformers" = 2 tokens. |
| **Next-token prediction** | The LLM predicts the most likely next token given everything before it |
| **Temperature** | Controls randomness. 0 = always picks the most likely token. 1.5 = adventurous |
| **Context window** | How much text the model can "see" at once. Its working memory. |
| **Hallucination** | When the model generates confident-sounding text that is factually wrong |

### The anatomy of an LLM prompt:

```
┌─────────────────────────────────────────────────────┐
│ SYSTEM PROMPT                                       │
│ "You are a helpful teaching assistant. Always       │
│  explain things simply. Avoid jargon."              │
│                                                     │
│ Sets the model's persona, rules, and constraints.   │
├─────────────────────────────────────────────────────┤
│ USER MESSAGE                                        │
│ "Explain what a neural network is."                 │
│                                                     │
│ The actual request or question.                     │
├─────────────────────────────────────────────────────┤
│ ASSISTANT RESPONSE (generated)                      │
│ "A neural network is like a chain of filters..."    │
└─────────────────────────────────────────────────────┘
```

In [5]:
# ── DEMO: Seeing temperature in action ───────────────────────────────────────
# Same prompt, different temperatures — watch the outputs change

prompt = "Continue this story in one sentence: The robot opened its eyes for the first time and"

print("📝 Prompt:", prompt)
print("\n" + "=" * 60)

for temp in [0.1, 0.7, 1.4]:
    response = call_llm(prompt, temperature=temp)
    print(f"\n🌡️  Temperature = {temp}:")
    print(f"   {response.strip()[:200]}")

print("\n" + "=" * 60)
print("🤔 How did the outputs differ? What does this tell you about temperature?")

Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📝 Prompt: Continue this story in one sentence: The robot opened its eyes for the first time and



Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🌡️  Temperature = 0.1:
   it marveled at the vibrant, unfamiliar world around it, eager to explore and learn about its new existence.


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🌡️  Temperature = 0.7:
   it was greeted by the vibrant colors and intricate details of a world it had only ever known through data and simulations.

🌡️  Temperature = 1.4:
   witnessed the breathtaking diversity of life and technology surrounding it in the sprawling, innovative metropolis.

🤔 How did the outputs differ? What does this tell you about temperature?


---
## Section 2 — Demonstrating Hallucination

Before we fix hallucination with RAG, you need to see it clearly.

We're going to ask the LLM about a **fictional event** that we made up — the "Bodhi Learning Festival". A real LLM trained on internet data has no information about it. Watch what happens.

In [6]:
# ── DEMO: Watch the LLM hallucinate ──────────────────────────────────────────
# Ask about something specific and fictional — observe confident wrong answers

hallucination_questions = [
    "How many attendees were at the Bodhi Learning Festival 2024 in Nepal?",
    "What is the NAIS and how many members does it have?",
    "What is NepaliBERT and what was it trained on?"
]

print("🔍 HALLUCINATION DEMO — LLM with NO document context")
print("=" * 60)
print("⚠️  The answers below may be WRONG — watch for confident-sounding errors\n")

for q in hallucination_questions:
    print(f"❓ Question: {q}")
    response = call_llm(q, temperature=0.3)
    print(f"🤖 LLM Answer: {response.strip()[:300]}")
    print("-" * 50 + "\n")

print("📌 Remember these answers. We'll compare them after building RAG.")

Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔍 HALLUCINATION DEMO — LLM with NO document context
⚠️  The answers below may be WRONG — watch for confident-sounding errors

❓ Question: How many attendees were at the Bodhi Learning Festival 2024 in Nepal?


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 LLM Answer: I am an AI and do not have real-time data or the ability to access current events or their details. Therefore, I cannot provide the exact number of attendees for the Bodhi Learning Festival 2024 in Nepal. Please check the official website or latest news sources for the most accurate information.
--------------------------------------------------

❓ Question: What is the NAIS and how many members does it have?


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 LLM Answer: The NAIS stands for the National Association of Insurance Commissioners. It is a U.S.-based organization that represents the state insurance regulatory agencies. The NAIS works to promote uniformity in insurance regulation across the United States, ensuring consumer protection, financial stability, 
--------------------------------------------------

❓ Question: What is NepaliBERT and what was it trained on?
🤖 LLM Answer: NepaliBERT is a transformer-based natural language processing model specifically designed for the Nepali language. It was developed by researchers at the Tribhuvan University, Nepal, to improve the performance of various natural language processing tasks on the Nepali language, which has limited res
--------------------------------------------------

📌 Remember these answers. We'll compare them after building RAG.


**Key observation:** The LLM sounds confident even when it's wrong. It doesn't say "I don't know" — it generates plausible-sounding text that fits the question. This is hallucination.

**Why it happens:** The model is optimized to produce fluent, coherent text. It was never trained to say "I don't have enough information." It always generates *something*.

**The fix:** Give the model the actual information as part of the prompt. That's what RAG does.

---
## Section 3 — 🟡 Prompt Engineering Exercises

Before we build RAG, spend 10 minutes mastering prompt structure. This is a critical skill — the quality of your prompts determines the quality of your outputs.

**Your goal:** Complete the same task 3 different ways and compare the outputs.

In [7]:
# ── EXERCISE 3A: System prompt changes everything ────────────────────────────
# Same user message — 3 different system prompts — 3 very different outputs

user_message = "Explain what machine learning is."

system_prompts = {
    "Default": "You are a helpful assistant.",
    "Teacher for children": "You are a patient teacher explaining concepts to 10-year-old students. Use simple words, short sentences, and fun analogies. No technical jargon.",
    "Senior engineer": "You are explaining to a senior software engineer with 10 years of experience. Be technical and precise. Skip basic definitions. Focus on implementation implications."
}

print(f"📝 User message: '{user_message}'")
print("=" * 60)

for persona, sys_prompt in system_prompts.items():
    response = call_llm(user_message, system_prompt=sys_prompt, temperature=0.5)
    print(f"\n🎭 System: '{persona}'")
    print(f"{response.strip()[:400]}")
    print("-" * 50)

print("\n🤔 How much did the system prompt change the response? What was most different?")

Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📝 User message: 'Explain what machine learning is.'


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🎭 System: 'Default'
Machine learning is a subset of artificial intelligence (AI) that focuses on the development of computer programs that can learn and improve their performance on a specific task over time, without being explicitly programmed. Instead of following fixed rules, machine learning algorithms use statistical techniques to analyze and identify patterns in data, enabling them to make predictions or decisi
--------------------------------------------------


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🎭 System: 'Teacher for children'
Imagine you have a really smart friend who loves to learn new things. One day, you show them lots of pictures of cats and dogs, and you tell them, "This is a cat, and this is a dog." Your friend looks at all these pictures many times and starts to understand the differences between a cat and a dog. 

Now, if you show your friend a new picture, they can probably guess whether it's a cat or a dog ju
--------------------------------------------------

🎭 System: 'Senior engineer'
Machine learning (ML) is a subset of artificial intelligence (AI) focused on building systems that can learn from and make predictions or decisions based on data, without being explicitly programmed for each specific task. From an implementation standpoint, this involves several critical technical considerations:

1. **Data Preparation**: The foundation of any ML project is high-quality, relevant 
--------------------------------------------------

🤔 How much did the system prompt

In [8]:
# ── EXERCISE 3B: Format control ──────────────────────────────────────────────
# Tell the model exactly what format you want — it will follow instructions

topic = "the difference between supervised and unsupervised learning"

format_prompts = [
    f"Explain {topic} in exactly 2 sentences. No more, no less.",
    f"Explain {topic} as a comparison table with 3 rows: Definition, Example, Use case.",
    f"Explain {topic} as if you are writing a tweet. Max 280 characters.",
]

print(f"🎯 Topic: '{topic}'")
print("=" * 60)

for i, prompt in enumerate(format_prompts, 1):
    response = call_llm(prompt, temperature=0.3)
    print(f"\n📋 Format {i}: '{prompt[:60]}...'")
    print(f"{response.strip()[:400]}")
    print("-" * 50)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🎯 Topic: 'the difference between supervised and unsupervised learning'


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📋 Format 1: 'Explain the difference between supervised and unsupervised l...'
Supervised learning uses labeled data to train models, predicting outcomes based on input data, whereas unsupervised learning works with unlabeled data, identifying patterns and structures without predefined outcomes. The key distinction lies in the presence of labeled data in supervised learning versus its absence in unsupervised learning.
--------------------------------------------------


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📋 Format 2: 'Explain the difference between supervised and unsupervised l...'
| Aspect         | Supervised Learning                                                                 | Unsupervised Learning                                                                 |
|----------------|------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------|
| **Definition**
--------------------------------------------------

📋 Format 3: 'Explain the difference between supervised and unsupervised l...'
Supervised learning uses labeled data to predict outcomes, while unsupervised learning finds patterns in unlabeled data. #MachineLearning #DataScience
--------------------------------------------------


In [9]:
# 🟡 YOUR TURN — Write your own prompt variations
# Pick any AI topic and try 3 different prompt styles
# ─────────────────────────────────────────────────────────────────────────────

your_topic = "neural networks"  # ← change this

your_prompts = [
    f"Explain {your_topic} in a haiku.",                   # ← modify these
    f"Explain {your_topic} using a cooking analogy.",
    f"List 5 surprising facts about {your_topic}.",
]

# ─────────────────────────────────────────────────────────────────────────────

print(f"🧪 Your experiments on: '{your_topic}'")
print("=" * 60)

for i, prompt in enumerate(your_prompts, 1):
    response = call_llm(prompt, temperature=0.7)
    print(f"\n🔬 Prompt {i}: '{prompt}'")
    print(f"{response.strip()[:500]}")
    print("-" * 50)

print("\n💡 Which prompt gave the most useful output? Why?")

Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧪 Your experiments on: 'neural networks'


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔬 Prompt 1: 'Explain neural networks in a haiku.'
Neurons connect deep,
Learning from input streams,
Models grow, achieve.
--------------------------------------------------


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔬 Prompt 2: 'Explain neural networks using a cooking analogy.'
Imagine you're trying to teach a group of chefs (neurons) how to make a complex dish (solve a problem or make a prediction). Each chef has their own specialty (function) they're good at, like chopping vegetables, sautéing ingredients, or baking. 

Now, when you put these chefs together to create the dish, they need to work in a coordinated way. Some chefs might need to start by chopping vegetables, while others focus on combining ingredients or adding flavor with spices. The chefs communicate wi
--------------------------------------------------

🔬 Prompt 3: 'List 5 surprising facts about neural networks.'
1. Neural networks can learn to play games without being explicitly programmed: One of the most surprising facts about neural networks is their ability to learn and improve at games like Go, Chess, and video games without any prior knowledge or explicit programming. They learn by trial and error, receiving feedback from 

---
## Section 4 — Building the RAG Pipeline

Now we fix hallucination. We'll build a complete RAG system step by step.

### The 4 steps we're about to implement:

```
STEP 1: CHUNK     — Split the document into smaller pieces
STEP 2: EMBED     — Convert each chunk to a vector (list of numbers)
STEP 3: STORE     — Save vectors in ChromaDB for fast retrieval
STEP 4: RETRIEVE  — Given a question, find matching chunks
       + GENERATE — Give those chunks to the LLM as context
```

In [10]:
# ── STEP 1: LOAD AND CHUNK THE DOCUMENT ──────────────────────────────────────
# We split the document into smaller pieces so we can retrieve
# only the relevant parts (not the whole document) for each question

import os

def load_text_file(filepath):
    """Load a text file and return its contents."""
    with open(filepath, 'r', encoding='utf-8') as f:
        return f.read()

def chunk_text(text, chunk_size=400, overlap=50):
    """
    Split text into overlapping chunks.

    chunk_size: roughly how many words per chunk
    overlap: how many words to repeat between adjacent chunks
             (overlap prevents losing context at chunk boundaries)
    """
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # step forward with overlap

    return chunks

# ── Load the document ────────────────────────────────────────────────────────
# Option A: use the provided file (uploaded with notebook)
# Option B: paste text directly

# Try loading the file, fall back to pasting text if file not found
DOCUMENT_PATH = "/content/nepal_facts (2).txt"   # ← change to your file path

if os.path.exists(DOCUMENT_PATH):
    document_text = load_text_file(DOCUMENT_PATH)
    print(f"✅ Loaded document: {DOCUMENT_PATH}")
    print(f"   Total characters: {len(document_text):,}")
    print(f"   Total words: {len(document_text.split()):,}")
else:
    print(f"⚠️  File '{DOCUMENT_PATH}' not found.")
    print("   Paste your document text in the cell below instead.")
    document_text = ""

✅ Loaded document: /content/nepal_facts (2).txt
   Total characters: 6,311
   Total words: 933


In [11]:
# If you don't have the file, paste your document text here instead
# ─────────────────────────────────────────────────────────────────────────────
# Only run this cell if the file wasn't found above

# document_text = """
# Paste your document text here.
# It can be anything: a Wikipedia article, your lecture notes,
# a news article, a story, anything with information in it.
# """

print("(Run this cell only if the file wasn't found above — uncomment the text above)")

(Run this cell only if the file wasn't found above — uncomment the text above)


In [12]:
# Chunk the document and inspect the result
chunks = chunk_text(document_text, chunk_size=300, overlap=40)

print(f"📄 Document chunked into {len(chunks)} pieces")
print(f"   Each chunk: ~300 words with 40-word overlap\n")

print("Preview of first 2 chunks:")
print("=" * 60)
for i, chunk in enumerate(chunks[:2]):
    words = len(chunk.split())
    print(f"\n[CHUNK {i+1}] ({words} words)")
    print(chunk[:300] + "..." if len(chunk) > 300 else chunk)
    print("-" * 40)

print("\n💡 Notice: chunks overlap slightly — this prevents losing context")
print("   that spans a chunk boundary.")

📄 Document chunked into 4 pieces
   Each chunk: ~300 words with 40-word overlap

Preview of first 2 chunks:

[CHUNK 1] (300 words)
NEPAL AI RESEARCH INSTITUTE — KNOWLEDGE BASE DOCUMENT Version 1.0 | Created for Workshop Demo Purposes ======================================================= SECTION 1: NEPAL — GEOGRAPHY AND OVERVIEW Nepal is a landlocked country in South Asia, bordered by China to the north and India to the south,...
----------------------------------------

[CHUNK 2] (300 words)
Tika, a ceremony between brothers and sisters. SECTION 3: THE NEPAL ARTIFICIAL INTELLIGENCE SOCIETY (NAIS) The Nepal Artificial Intelligence Society (NAIS) was established in 2021 in Kathmandu with the aim of promoting AI research, education, and policy development in Nepal. NAIS currently has 340 r...
----------------------------------------

💡 Notice: chunks overlap slightly — this prevents losing context
   that spans a chunk boundary.


In [13]:
# ── STEP 2: EMBED — Convert text to vectors ───────────────────────────────────
# Each chunk becomes a list of 384 numbers that captures its "meaning"
# Similar chunks will have similar vectors

from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')  # fast, 384-dimensional

# Embed a sample to show what a vector looks like
sample_text = "Nepal is a landlocked country in South Asia."
sample_vector = embed_model.encode(sample_text)

print(f"\n✅ Embedding model loaded")
print(f"\n📊 What an embedding looks like:")
print(f"   Input text: '{sample_text}'")
print(f"   Output vector shape: {sample_vector.shape}")
print(f"   First 10 dimensions: {sample_vector[:10].round(4)}")
print(f"\n💡 These 384 numbers encode the 'meaning' of the sentence.")
print(f"   Similar sentences → similar numbers → close in vector space")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✅ Embedding model loaded

📊 What an embedding looks like:
   Input text: 'Nepal is a landlocked country in South Asia.'
   Output vector shape: (384,)
   First 10 dimensions: [-0.0026  0.0613 -0.0517  0.0416  0.0513 -0.0468  0.0036 -0.063  -0.0037
 -0.0336]

💡 These 384 numbers encode the 'meaning' of the sentence.
   Similar sentences → similar numbers → close in vector space


In [14]:
# Quick demo: semantic similarity using embeddings
# Show that similar sentences produce similar vectors (high cosine similarity)

import numpy as np

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

sentences = [
    "Nepal is a beautiful country in South Asia.",       # similar to anchor
    "The Himalayan nation borders China and India.",     # similar to anchor
    "I love eating pizza on Friday evenings.",           # unrelated
    "Machine learning requires lots of training data."   # unrelated
]
anchor = "Nepal is a landlocked country in South Asia."

anchor_vec = embed_model.encode(anchor)

print(f"📍 Anchor sentence: '{anchor}'")
print("\nSimilarity to other sentences:")
print("-" * 55)

for sent in sentences:
    vec = embed_model.encode(sent)
    sim = cosine_similarity(anchor_vec, vec)
    bar = '█' * int(sim * 30)
    print(f"{sim:.3f} {bar}")
    print(f"      '{sent[:60]}'")

print("\n💡 High similarity = same topic. Low similarity = unrelated.")
print("   This is how retrieval works — find the chunks closest to the question.")

📍 Anchor sentence: 'Nepal is a landlocked country in South Asia.'

Similarity to other sentences:
-------------------------------------------------------
0.756 ██████████████████████
      'Nepal is a beautiful country in South Asia.'
0.555 ████████████████
      'The Himalayan nation borders China and India.'
-0.031 
      'I love eating pizza on Friday evenings.'
0.036 █
      'Machine learning requires lots of training data.'

💡 High similarity = same topic. Low similarity = unrelated.
   This is how retrieval works — find the chunks closest to the question.


In [15]:
# ── STEP 3: STORE — Save vectors in ChromaDB ──────────────────────────────────
# ChromaDB is a vector database: it stores text + its embedding vector
# and can quickly find the most similar vectors to a query

import chromadb

# Create an in-memory ChromaDB database (no files, resets when notebook restarts)
chroma_client = chromadb.Client()

# Delete collection if it already exists (for clean re-runs)
try:
    chroma_client.delete_collection("workshop_docs")
except:
    pass

collection = chroma_client.create_collection("workshop_docs")

print("Embedding all chunks and storing in ChromaDB...")
print(f"(Embedding {len(chunks)} chunks — takes ~10-30 seconds)")

# Embed all chunks
chunk_embeddings = embed_model.encode(chunks, show_progress_bar=True)

# Store in ChromaDB with unique IDs
collection.add(
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"\n✅ {len(chunks)} chunks stored in ChromaDB")
print(f"   Each chunk: stored as text + 384-dimensional vector")
print(f"   Total: {collection.count()} entries in the collection")

Embedding all chunks and storing in ChromaDB...
(Embedding 4 chunks — takes ~10-30 seconds)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ 4 chunks stored in ChromaDB
   Each chunk: stored as text + 384-dimensional vector
   Total: 4 entries in the collection


In [16]:
# ── STEP 4A: RETRIEVE — Find relevant chunks for a question ───────────────────
# This is the retrieval half of RAG

def retrieve_chunks(question, n_results=3):
    """
    Given a question, find the most relevant document chunks.

    Process:
    1. Embed the question (same model as the document)
    2. Find the n_results chunks with highest cosine similarity
    3. Return those chunks as text
    """
    question_embedding = embed_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )

    retrieved = results['documents'][0]  # list of matching chunks
    distances = results['distances'][0]   # lower = more similar

    return retrieved, distances

# Test retrieval — show what comes back for a sample question
test_question = "How many members does NAIS have?"
retrieved_chunks, distances = retrieve_chunks(test_question, n_results=2)

print(f"🔍 Question: '{test_question}'")
print(f"\n📦 Retrieved {len(retrieved_chunks)} most relevant chunks:")
print("=" * 60)

for i, (chunk, dist) in enumerate(zip(retrieved_chunks, distances)):
    similarity = 1 - dist  # convert distance to similarity
    print(f"\n[Chunk {i+1}] Relevance: {similarity:.3f}")
    print(chunk[:300] + "..." if len(chunk) > 300 else chunk)
    print("-" * 40)

print("\n💡 The retriever found these without reading the whole document.")
print("   Now we give these to the LLM as context.")

🔍 Question: 'How many members does NAIS have?'

📦 Retrieved 2 most relevant chunks:

[Chunk 1] Relevance: -0.160
Agriculture and NAIS have jointly developed a mobile application called "KrishiAI" (released in 2023) that allows farmers to photograph crop leaves and receive instant diagnosis of diseases and pest infestations. The model was trained on 45,000 annotated images across 18 crop species. SECTION 7: KEY...
----------------------------------------

[Chunk 2] Relevance: -0.342
Tika, a ceremony between brothers and sisters. SECTION 3: THE NEPAL ARTIFICIAL INTELLIGENCE SOCIETY (NAIS) The Nepal Artificial Intelligence Society (NAIS) was established in 2021 in Kathmandu with the aim of promoting AI research, education, and policy development in Nepal. NAIS currently has 340 r...
----------------------------------------

💡 The retriever found these without reading the whole document.
   Now we give these to the LLM as context.


In [17]:
# ── STEP 4B: GENERATE — Use retrieved context to answer the question ──────────
# This is the generation half of RAG

def rag_answer(question, n_chunks=3, temperature=0.3):
    """
    Answer a question using RAG:
    1. Retrieve relevant chunks from the document
    2. Build a prompt with the chunks as context
    3. Ask the LLM to answer using only that context
    """
    # Step 1: Retrieve relevant chunks
    chunks_retrieved, _ = retrieve_chunks(question, n_results=n_chunks)

    # Step 2: Build the context string
    context = "\n\n---\n\n".join(chunks_retrieved)

    # Step 3: Build the RAG prompt
    rag_prompt = f"""Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I don't have that information in the provided document."
Do not make up information. Be specific and cite details from the context.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    # Step 4: Generate the answer
    answer = call_llm(
        rag_prompt,
        system_prompt="You are a precise document assistant. Only answer from the provided context.",
        temperature=temperature
    )

    return answer.strip(), chunks_retrieved

print("✅ RAG system ready.")
print("Run the next cell to see it in action.")

✅ RAG system ready.
Run the next cell to see it in action.


---
## Section 5 — 🔍 The Key Demo: With vs Without RAG

This is the moment everything clicks. We'll ask the same questions both ways and compare the answers.

In [18]:
# ── THE CRITICAL COMPARISON ───────────────────────────────────────────────────
# Same question. Same model. Completely different answers.

comparison_questions = [
    "How many attendees were at the Bodhi Learning Festival 2024?",
    "What is NAIS and how many members does it have?",
    "What is NepaliBERT and what corpus was it trained on?"
]

for question in comparison_questions:
    print("\n" + "=" * 65)
    print(f"❓ QUESTION: {question}")
    print("=" * 65)

    # WITHOUT RAG
    print("\n❌ WITHOUT RAG (LLM alone):")
    without_rag = call_llm(question, temperature=0.3)
    print(without_rag.strip()[:400])

    # WITH RAG
    print("\n✅ WITH RAG (document-grounded):")
    with_rag, retrieved = rag_answer(question)
    print(with_rag[:400])

    print("\n" + "-" * 65)

print("\n🎯 Key insight: Same model. Same parameters. Only the context changed.")
print("   RAG doesn't change the LLM — it changes what information the LLM sees.")

Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



❓ QUESTION: How many attendees were at the Bodhi Learning Festival 2024?

❌ WITHOUT RAG (LLM alone):


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I am an AI and do not have real-time access to current or future event data. Therefore, I am unable to provide the exact number of attendees for the Bodhi Learning Festival 2024. Please check the official website or contact the event organizers for the most accurate information.

✅ WITH RAG (document-grounded):


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

NAIS stands for the North American Indian Chiefs, also known as the National Indian Brotherhood (NIB) or the Assembly of First Nations (AFN) in its current form. It is a national organization representing First Nations people in Canada. The exact number of members can vary over time as new members are added and others may leave or pass away. As of my last update in 2021, the organization reported 

✅ WITH RAG (document-grounded):


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The Nepal Artificial Intelligence Society (NAIS) was established in 2021 in Kathmandu with the aim of promoting AI research, education, and policy development in Nepal. NAIS currently has 340 registered members spanning academia, government, and private industry.

-----------------------------------------------------------------

❓ QUESTION: What is NepaliBERT and what corpus was it trained on?

❌ WITHOUT RAG (LLM alone):


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


NepaliBERT is a bidirectional Transformer-based language model specifically designed for the Nepali language. It was developed to improve natural language processing tasks in the Nepali language, such as text classification, named entity recognition, and question answering. 

The model was trained on a large corpus of Nepali text data, which includes a wide range of sources such as news articles, 

✅ WITH RAG (document-grounded):
NepaliBERT is a BERT-based language model released by the Nepali NLP Consortium in 2022. It was trained on a corpus of 15 million Nepali sentences.

-----------------------------------------------------------------

🎯 Key insight: Same model. Same parameters. Only the context changed.
   RAG doesn't change the LLM — it changes what information the LLM sees.


---
## Section 6 — 🟡 Exercise: Build Your Own RAG Chatbot

Now it's your turn. You'll swap in your own document and run the full RAG pipeline against it.

**Steps:**
1. Find any text document (Wikipedia article, news article, your notes — anything)
2. Save it as a `.txt` file and upload it to Colab, OR paste it directly
3. Run the pipeline against it
4. Write 5 questions — some the document can answer, some it can't

In [19]:
# 🟡 LOAD YOUR OWN DOCUMENT
# Option A: Upload a .txt file to Colab (Files panel on the left) and change the path
# Option B: Paste the text directly below
# ─────────────────────────────────────────────────────────────────────────────

# Option A: file path
your_doc_path = "your_document.txt"   # ← change this

# Option B: paste text (uncomment and fill in)
# your_document_text = """
# Paste your document text here.
# Make sure it's at least 300 words for meaningful chunking.
# """

# ─────────────────────────────────────────────────────────────────────────────

# Load whichever option you chose
if os.path.exists(your_doc_path):
    your_document_text = load_text_file(your_doc_path)
    print(f"✅ Loaded: {your_doc_path} ({len(your_document_text.split())} words)")
elif 'your_document_text' in dir():
    print(f"✅ Using pasted text ({len(your_document_text.split())} words)")
else:
    print("⚠️  No document found. Using the default Nepal document.")
    your_document_text = document_text

⚠️  No document found. Using the default Nepal document.


In [20]:
# Build a new ChromaDB collection for your document

try:
    chroma_client.delete_collection("my_docs")
except:
    pass

my_collection = chroma_client.create_collection("my_docs")

# Chunk and embed
my_chunks = chunk_text(your_document_text, chunk_size=300, overlap=40)
print(f"Chunked into {len(my_chunks)} pieces. Embedding...")

my_embeddings = embed_model.encode(my_chunks, show_progress_bar=True)

my_collection.add(
    documents=my_chunks,
    embeddings=my_embeddings.tolist(),
    ids=[f"mychunk_{i}" for i in range(len(my_chunks))]
)

print(f"\n✅ Your document is indexed. Ready to answer questions.")

Chunked into 4 pieces. Embedding...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Your document is indexed. Ready to answer questions.


In [21]:
# 🟡 ASK YOUR OWN QUESTIONS
# ─────────────────────────────────────────────────────────────────────────────

def my_rag_answer(question, n_chunks=3):
    """RAG using YOUR document collection."""
    q_emb = embed_model.encode(question).tolist()
    results = my_collection.query(query_embeddings=[q_emb], n_results=n_chunks)
    context = "\n\n---\n\n".join(results['documents'][0])

    prompt = f"""Answer using ONLY the context below. If not in the context, say so.

CONTEXT:
{context}

QUESTION: {question}
ANSWER:"""
    return call_llm(prompt, temperature=0.2)

# Write 5 questions about your document
# 3 that the document CAN answer, 2 that it CANNOT
your_questions = [
    "Question 1 — something your document contains",
    "Question 2 — another thing your document contains",
    "Question 3 — a specific fact or number in your document",
    "Question 4 — something NOT in your document (should say 'I don't have that')",
    "Question 5 — another thing NOT in your document",
]

# ─────────────────────────────────────────────────────────────────────────────

for q in your_questions:
    if "Question" in q:
        print(f"⚠️  Replace placeholder: '{q}'")
        continue
    print(f"\n❓ {q}")
    answer = my_rag_answer(q)
    print(f"🤖 {answer.strip()[:400]}")
    print("-" * 50)

⚠️  Replace placeholder: 'Question 1 — something your document contains'
⚠️  Replace placeholder: 'Question 2 — another thing your document contains'
⚠️  Replace placeholder: 'Question 3 — a specific fact or number in your document'
⚠️  Replace placeholder: 'Question 4 — something NOT in your document (should say 'I don't have that')'
⚠️  Replace placeholder: 'Question 5 — another thing NOT in your document'


---
## Section 7 — Interactive Chat Mode

Let's make this feel like a real chatbot — you type a question, it answers from your document.

In [22]:
# ── INTERACTIVE CHAT LOOP ─────────────────────────────────────────────────────
# Type your questions one at a time
# Type 'quit' to exit

print("💬 RAG CHATBOT — Powered by your document")
print("Type a question and press Enter. Type 'quit' to stop.")
print("=" * 55)

while True:
    try:
        user_input = input("\nYou: ").strip()
    except EOFError:
        break

    if not user_input:
        continue
    if user_input.lower() in ['quit', 'exit', 'q', 'stop']:
        print("\nChat ended. Great work! 🎉")
        break

    print("Bot: thinking...", end='\r')
    answer = my_rag_answer(user_input)
    print(f"Bot: {answer.strip()}")

💬 RAG CHATBOT — Powered by your document
Type a question and press Enter. Type 'quit' to stop.

You: who is balen shah


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: There is no information about a person named Balen Shah in the provided context.

You: what is the document about


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: The document provides an overview of various aspects related to artificial intelligence (AI) in Nepal, including language and NLP challenges, computer vision applications, key statistics, the role of the Nepal Artificial Intelligence Society (NAIS), AI education, and specific initiatives such as the Rural AI Initiative and the Bodhi Learning Festival (described as fictional for demonstration purposes).

You: quit

Chat ended. Great work! 🎉


---
## Section 8 — 🟡 Go Further (If You're Ahead)

**Challenge A — Show the retrieved chunks:**
Modify `my_rag_answer` to also print which chunks were retrieved. Do the retrieved chunks actually contain the answer?

**Challenge B — Add a confidence signal:**
Add a step that uses the retrieval similarity score to decide whether to answer or say "I'm not sure" (e.g., if max similarity < 0.3, the question is probably not in the document).

**Challenge C — Multi-document RAG:**
Add a second document to the same ChromaDB collection. Ask questions that require combining information from both. Does it work?

**Challenge D — System prompt experiments:**
Change the RAG system prompt to make the bot respond differently: as a Socratic teacher that asks back, as a journalist summarizing facts, or in bullet points only.

In [23]:
# Challenge A starter — show retrieved chunks alongside the answer

def rag_with_sources(question, n_chunks=3):
    """RAG that also shows which parts of the document were used."""
    q_emb = embed_model.encode(question).tolist()
    results = my_collection.query(
        query_embeddings=[q_emb],
        n_results=n_chunks,
        include=['documents', 'distances']
    )

    retrieved_docs = results['documents'][0]
    distances = results['distances'][0]

    context = "\n\n---\n\n".join(retrieved_docs)
    prompt = f"""Answer using ONLY the context. If not found, say so.

CONTEXT:
{context}

QUESTION: {question}
ANSWER:"""

    answer = call_llm(prompt, temperature=0.2)

    # Print the answer AND the sources
    print(f"🤖 ANSWER: {answer.strip()}")
    print(f"\n📚 SOURCES USED ({len(retrieved_docs)} chunks):")
    for i, (doc, dist) in enumerate(zip(retrieved_docs, distances)):
        sim = 1 - dist
        print(f"  [{i+1}] Relevance: {sim:.3f} | '{doc[:100]}...'")

# Test it
rag_with_sources("What is NepaliBERT?")

Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 ANSWER: NepaliBERT is a BERT-based language model pre-trained on a corpus of 15 million Nepali sentences, released by the Nepali NLP Consortium in 2022.

📚 SOURCES USED (3 chunks):
  [1] Relevance: 0.085 | 'RAG.) SECTION 5: LANGUAGE AND NLP CHALLENGES IN NEPAL Nepal has over 120 spoken languages. Nepali (D...'
  [2] Relevance: -0.151 | 'NEPAL AI RESEARCH INSTITUTE — KNOWLEDGE BASE DOCUMENT Version 1.0 | Created for Workshop Demo Purpos...'
  [3] Relevance: -0.215 | 'Agriculture and NAIS have jointly developed a mobile application called "KrishiAI" (released in 2023...'


---
## Section 9 — Day 2 Recap

Before you close this notebook, make sure you can answer these:

1. **What does an LLM actually do when it generates text?**
   > *Predicts the next most likely token, given everything before it. Repeats this until done. It's a statistical pattern matcher, not a knowledge database.*

2. **Why do LLMs hallucinate?**
   > *They're optimized to produce fluent, confident-sounding text — not to accurately signal uncertainty. They generate plausible completions even when they have no reliable information.*

3. **What are the 4 steps of a RAG pipeline?**
   > *Chunk → Embed → Store → Retrieve + Generate.*

4. **What does an embedding do?**
   > *Converts text into a fixed-length vector of numbers that captures semantic meaning. Similar text → similar vectors → close in vector space.*

5. **What is ChromaDB doing in this pipeline?**
   > *Storing vectors and enabling fast similarity search — given a question vector, find the document vectors that are closest in meaning.*

---

## 🚀 Coming Tomorrow — Day 3: Computer Vision & Model Training

Tomorrow you'll:
- **Annotate your own image dataset** (bring ideas — what do you want to detect?)
- **Train a custom YOLOv8 object detector** on that data — in ~10 minutes
- Understand transfer learning: why you don't need 1 million images
- Evaluate your model and understand what the metrics mean

> **Optional prep:** Create a free account at https://roboflow.com — it's where your data will live.

> **Think:** What object, gesture, or visual category would you want a computer to detect for you? Bring 3 ideas.

**See you tomorrow. 🎯**